# Conversational BI: Semantic Views + Cortex Analyst + Agent

Build one governed definition of churn, revenue, and engagement with a semantic view, query it with `SEMANTIC_VIEW()`, search the CX chat telemetry, and combine both in a Cortex Agent.

### What You'll Learn

- Inspect a semantic view's metrics and dimensions
- Query governed metrics with `SEMANTIC_VIEW()` (no JOINs, no re-defined logic)
- Search unstructured chat telemetry with Cortex Search
- Combine structured metrics + unstructured conversations in a Cortex Agent

**Prerequisites:** Run `lab/setup.sql` first. For the search/agent sections, also run the
**AI Functions** module (`cx-ai-functions/lab`) so `AI_FUNCTIONS.CHAT_THREADS` exists.

---
## Section 1: Connect & Inspect the Semantic View

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FIELD_CX_DEMO").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
session.sql("USE WAREHOUSE CONVERSATIONAL_BI_WH").collect()
print("Connected to FIELD_CX_DEMO.ANALYTICS")

In [ ]:
# The governed metrics available in the semantic view
session.sql("SHOW SEMANTIC METRICS IN VIEW CX_ANALYTICS_SV").show(max_width=120)

---
## Section 2: Query Governed Metrics with SEMANTIC_VIEW()

Pick metrics and dimensions — the joins and metric logic are already defined in the view.

In [ ]:
# Churn rate by plan — one governed definition, no JOINs
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        CX_ANALYTICS_SV
        METRICS churn_rate, customer_count
        DIMENSIONS customers.plan
    )
    ORDER BY customers.plan
""").show()

In [ ]:
# Total MRR and average home value by customer segment
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        CX_ANALYTICS_SV
        METRICS total_mrr, avg_home_value, customer_count
        DIMENSIONS customers.customer_type
    )
    ORDER BY customers.customer_type
""").show()

---
## Section 3: Search the Unstructured CX Telemetry

`CHAT_SEARCH` (created in `setup.sql`) indexes the chat threads from the AI Functions module.
Preview it with `SEARCH_PREVIEW`. Requires the AI Functions module to have been set up.

In [ ]:
# Find chats about cancellations / refunds
session.sql("""
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CHAT_SEARCH',
            '{
               "query": "customer wants to cancel or get a refund",
               "columns": ["transcript", "customer_id"],
               "limit": 5
            }'
        )
    )['results'] AS results
""").show(max_width=200)

---
## Section 4: Combine Both in the Cortex Agent

`CX_INTELLIGENCE_AGENT` (created in `setup.sql`) has two tools: Cortex Analyst over the
semantic view and Cortex Search over the chat telemetry. Chat with it in Snowsight
(**AI & ML → Agents**), or run it programmatically with `DATA_AGENT_RUN`.

Try: *"Which churn-risk customers had negative support chats?"* — the agent uses Analyst for
the metric and Search for the conversations, in one answer.

In [ ]:
# Confirm the agent exists; then chat with it in Snowsight (AI & ML -> Agents)
session.sql("SHOW AGENTS LIKE 'CX_INTELLIGENCE_AGENT'").show(max_width=120)

# The agent combines:
#   - CX_Metrics      (Cortex Analyst over CX_ANALYTICS_SV)   -> churn, revenue, engagement
#   - Chat_Telemetry  (Cortex Search over CHAT_SEARCH)        -> what customers said
# Extend it with custom tools (stored procs/UDFs) or MCP connectors to fit your workflow.

---
## Summary

You built a conversational BI stack on one governed foundation:

- **Semantic view** — centralized churn / revenue / engagement definitions
- **`SEMANTIC_VIEW()`** — governed metrics with no re-written JOINs or logic
- **Cortex Search** — the unstructured CX chat telemetry, searchable
- **Cortex Agent** — combines the semantic view (Analyst) and chat telemetry (Search) in one assistant, extensible with custom tools and MCP

Define the business once, and let Cortex Analyst, the agent, and BI tools like Sigma all
speak the same language. Pairs with the **AI Functions** module, which produces the chat
telemetry this agent searches.